# Numerical Comparison of Restart Methods of the Arnoldi Method

## Imports

In [ ]:
using LinearAlgebra
using JacobiDavidson
using LinearMaps
using MatrixDepot
ENV["GKSwstype"] = "nul"
using Plots
using ProgressMeter
using LaTeXStrings
using BenchmarkTools

include("../src/Orthogonalization.jl")
include("../src/Arnoldi.jl")
include("../src/ImplicitRestart.jl")
include("../src/Eigenpairs.jl")
include("../src/BadRestart.jl")

## Experiments

We use the large sparse matrix "rajat12" from MatrixDepot to compare the time and memory allocation cost of Arnoldi, Naive Restart and IRAM as a function of the subspace dimension.

In [ ]:
A = matrixdepot(r"rajat12") + 10 * I
n = size(A, 1)
num_eig = 4

display(A)

### Memory and Time

In [ ]:
subspace_grid = Int.(range(6, 50, 23))
iterations = 10

memory_arnoldi = []
memory_naive = []
memory_iram = []

time_arnoldi = []
time_naive = []
time_iram = []

@showprogress for subspace_dim in subspace_grid
    benchmark_arnoldi = @benchmark Eigenpairs.eigenpairs_arnoldi($A, $n, num_eig=$num_eig, subspace_dim=$subspace_dim, tol=0) samples=iterations evals=iterations
    push!(memory_arnoldi, benchmark_arnoldi.memory)
    push!(time_arnoldi, median(benchmark_arnoldi).time)
    benchmark_naive = @benchmark Eigenpairs.eigenpairs_naive_restart($A, $n, num_eig=$num_eig, subspace_dim=$subspace_dim, restart_dim=max($num_eig, Int(floor($subspace_dim/2))), tol=0) samples=iterations evals=iterations
    push!(memory_naive, benchmark_naive.memory)
    push!(time_naive, median(benchmark_naive).time)
    benchmark_iram = @benchmark Eigenpairs.eigenpairs_iram($A, $n, num_eig=$num_eig, subspace_dim=$subspace_dim, restart_dim=max($num_eig, Int(floor($subspace_dim/2))), tol=0) samples=iterations evals=iterations
    push!(memory_iram, benchmark_iram.memory)
    push!(time_iram, median(benchmark_iram).time)
end

memory_arnoldi = memory_arnoldi ./ 1024^2
memory_naive = memory_naive ./ 1024^2
memory_iram = memory_iram ./ 1024^2

time_arnoldi = time_arnoldi ./ 1e9
time_naive = time_naive ./ 1e9
time_iram = time_iram ./ 1e9;

In [ ]:
println("Extremum time:, Arnoldi: $(minimum(time_arnoldi))s, Naive: $(minimum(time_naive))s, IRAM: $(minimum(time_iram))s")
println("Extremum memory:, Arnoldi: $(minimum(memory_arnoldi))MB, Naive: $(minimum(memory_naive))MB, IRAM: $(minimum(memory_iram))MB")

println("Extremum time:, Arnoldi: $(maximum(time_arnoldi))s, Naive: $(maximum(time_naive))s, IRAM: $(maximum(time_iram))s")
println("Extremum memory:, Arnoldi: $(maximum(memory_arnoldi))MB, Naive: $(maximum(memory_naive))MB, IRAM: $(maximum(memory_iram))MB")

In [ ]:
println()
println("Memory Arnoldi: ", memory_arnoldi)
println()
println("Memory Naive: ", memory_naive)
println()
println("Memory IRAM: ", memory_iram)

println()
println("Time Arnoldi: ", time_arnoldi)
println()
println("Time Naive: ", time_naive)
println()
println("Time IRAM: ", time_iram)
println()

In [ ]:
p = plot(subspace_grid, memory_arnoldi, color=:blue,
     title = "Memory allocation vs Subspace dimension", xlabel="Krylov subspace dimension", ylabel="Memory allocated (MB)",
     xlim=[6, 50], ylim = [1e0, 1e7],
     xaxis=:log, yaxis=:log,
     label = "Arnoldi")

plot!(p, subspace_grid, memory_naive, color=:green,
     label = "Naive Restart")

plot!(p, subspace_grid, memory_iram, color=:red,
     label = "IRAM")

plot!(p, subspace_grid, subspace_grid .^ 2, color=:black, linestyle=:dash,
     label=L"O(k^2)")

savefig(p, "../fig/Arnoldi vs Naive vs IRAM/memory_vs_subspace.png")

In [ ]:
p = plot(subspace_grid, time_arnoldi, color=:blue,
     title = "Time vs Subspace dimension", xlabel="Krylov subspace dimension", ylabel="Time (s)",
     xlim=[6, 50], ylim = [1e-4, 1e3],
     xaxis=:log, yaxis=:log,
     label = "Arnoldi")

plot!(p, subspace_grid, time_naive, color=:green,
     label = "Naive Restart")

plot!(p, subspace_grid, time_iram, color=:red,
     label = "IRAM")
     
plot!(p, subspace_grid, subspace_grid .^ 2 / 1e3, color=:black, linestyle=:dash,
     label=L"O(k^2)")

savefig(p, "../fig/Arnoldi vs Naive vs IRAM/time_vs_subspace.png")